In [1]:
import torch
print(torch.__version__)


2.10.0+cu126


In [2]:
# device configuration for model training
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)


cuda


In [ ]:
# importing all necessary items for model training
from torch.utils.data import DataLoader

from models.model import SimSearchModel
from utils.dataset import SimSearchDataset
from utils.augmentations import SimSearch_transform
from losses.nt_xent import nt_xent_loss


# creating dataset for model training
dataset = SimSearchDataset(
    root_dir="data",
    transform=SimSearch_transform
)

# dataloader for loading the data in model training
loader = DataLoader(
    dataset=dataset,
    batch_size=32, # this is important stuffs
    shuffle=True,
    num_workers=4,
    drop_last=True
)




In [6]:
simsearchmodel = SimSearchModel().to(device=device)  # loading model into device

# optimizer for model training
optimizer=torch.optim.AdamW(
    simsearchmodel.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

EPOCHS=300 # total number of epochs for CIAR 10 dataset
# adding schedular
# contrastive learning use cosine decay
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)


In [7]:
# SimSearch model training
def SimSearch_training(model, loader, loss_fn, optimizer_fn, device, scheduler,epochs):
    
    for epoch in range(epochs):
        total_loss=0
        for view1, view2 in loader: # iterating through loader
            view1=view1.to(device) # loading view1 into device, similarly for view2
            view2=view2.to(device)

            z1=model(view1) # embedding for view1
            z2=model(view2) # embedding for view2

            loss=loss_fn(z1,z2) # calculating loss
            optimizer_fn.zero_grad() # zero gradient of optimizer
            loss.backward() # backpropagation
            optimizer_fn.step() # usign optimizer function
            total_loss+=loss.item()
        
        scheduler.step() # stepping schedular after each epoch
        print(f"Epoch: {epoch}| Loss: {total_loss/len(loader):.4f}")


SimSearch_training(model=simsearchmodel,loader=loader,loss_fn=nt_xent_loss,optimizer_fn=optimizer,device=device,scheduler=scheduler,epochs=EPOCHS)



Epoch: 0| Loss: 2.8404
Epoch: 1| Loss: 2.7333
Epoch: 2| Loss: 2.6908
Epoch: 3| Loss: 2.6754
Epoch: 4| Loss: 2.6722
Epoch: 5| Loss: 2.6422
Epoch: 6| Loss: 2.6415
Epoch: 7| Loss: 2.6365
Epoch: 8| Loss: 2.6254
Epoch: 9| Loss: 2.6142
Epoch: 10| Loss: 2.6132
Epoch: 11| Loss: 2.6051
Epoch: 12| Loss: 2.5954
Epoch: 13| Loss: 2.6003
Epoch: 14| Loss: 2.5944
Epoch: 15| Loss: 2.5792
Epoch: 16| Loss: 2.5829
Epoch: 17| Loss: 2.5851
Epoch: 18| Loss: 2.5835
Epoch: 19| Loss: 2.5745
Epoch: 20| Loss: 2.5713
Epoch: 21| Loss: 2.5754
Epoch: 22| Loss: 2.5710
Epoch: 23| Loss: 2.5652
Epoch: 24| Loss: 2.5681
Epoch: 25| Loss: 2.5679
Epoch: 26| Loss: 2.5610
Epoch: 27| Loss: 2.5612
Epoch: 28| Loss: 2.5639
Epoch: 29| Loss: 2.5585
Epoch: 30| Loss: 2.5544
Epoch: 31| Loss: 2.5562
Epoch: 32| Loss: 2.5600
Epoch: 33| Loss: 2.5467
Epoch: 34| Loss: 2.5460
Epoch: 35| Loss: 2.5435
Epoch: 36| Loss: 2.5459
Epoch: 37| Loss: 2.5466
Epoch: 38| Loss: 2.5387
Epoch: 39| Loss: 2.5329
Epoch: 40| Loss: 2.5401
Epoch: 41| Loss: 2.5398
Ep

In [8]:
torch.save(simsearchmodel.encoder.state_dict(),"SimSearch_0_encoder.pth") # saving the encoder of the model
#encoder generates embeddings of the images
